<a href="https://colab.research.google.com/github/elizebeth18/Learn-python/blob/main/Module1_FastAPI_LLM_Serving_1_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 — API Development & Application Serving with FastAPI

**Goal:** By end of this session, you will have a working FastAPI application that serves an LLM over a REST API — with streaming, validation, and auto-generated documentation.

---

## Course Vision

Every module in this course builds on the same application. Today you build the foundation — the API layer. In later modules you will containerise it, deploy it to AWS, add monitoring, and scale it.

---

## What This Session Covers

- What an API is and why every AI application needs one
- REST API concepts — endpoints, methods, request/response
- Build endpoints step by step
- Request and response validation with Pydantic
- Streaming responses — return text as it's generated
- Auto-generated API documentation


---
# Part 1 — What is an API?

## The Problem Without an API

You trained a model. It lives in a Python script on your machine. Nobody else can use it unless they run your script themselves — with your Python version, your dependencies, your files.

An API solves this. It wraps your model behind a web address. Anyone — a mobile app, a website, another service, a teammate — can send a request to that address and get a response. They don't need to know anything about your code.

```
Without API:
User → needs Python + your code + your model → runs script → gets answer

With API:
User → sends HTTP request to your URL → gets answer back
```

## REST APIs

REST is the most common style of web API. It uses standard HTTP methods:

| Method | What It Does | Example |
|--------|-------------|---------|
| GET | Retrieve data | Get list of models |
| POST | Send data, get result | Send a question, get an answer |
| PUT | Update something | Update a configuration |
| DELETE | Remove something | Delete a conversation |

For AI applications, almost everything is POST — you send a prompt, you get a response.

## Why FastAPI?

FastAPI has become the standard framework for serving ML models and AI applications because:

- It is very fast — built on async Python
- You write Python type hints and it automatically validates requests
- It generates interactive API documentation automatically — no extra work
- It supports streaming responses — essential for LLMs
- The code is clean and readable


---
# Part 2 — Setup

## Step 0 — Install

In [ ]:
!pip install fastapi uvicorn pydantic openai httpx nest-asyncio -q
print("Done")

Done


## Step 1 — Imports

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
import uvicorn
import nest_asyncio

# This allows FastAPI to run inside a Jupyter notebook
nest_asyncio.apply()

print("All imports ready")

All imports ready


Reading a URL:

Scheme => http:// or https://
Host => which machine it is hosted on (localhost / 127.0.0.1)
Port => which server to use (a machine can host multiple servers) -> :8000
Path => /model  or model/sentiment-analysis (thing you want to access on the server)
Query => Optional details (ex ?name=ram)


---
# Part 3 — Your First FastAPI Application

## How FastAPI Works  
### FastAPI connects a URL + HTTP method to a Python function.  
You create an `app` object. Then you define functions and decorate them with the HTTP method and path. FastAPI does the rest — routing, parsing, validation, documentation.

```python
@app.get("/hello")        ← this function handles GET requests to /hello
def say_hello():
    return {"message": "Hello"}
```

That's it. One decorator, one function, one endpoint.


In [ ]:
Internet -> http://localhost:8080 -> fast api application -> calls python function -> calls openai gpt -> returns response

SyntaxError: invalid syntax (3101898210.py, line 1)

## Step 2 — Create the App

In [ ]:
#FastAPI() only creates/configures the application.
#It does not start the web server. Uvicorn is typically used to start the server.

In [ ]:
app = FastAPI( #Creates the FastAPI application object.
    title="Enterprise GenAI API", #Name of your API, shown in the API documentation.
    description="AI-powered API built with FastAPI", #Short explanation of what your API does.
    version="1.0.0" #Version number of your API
)

print("FastAPI app created")

FastAPI app created


FastAPi

browser -> http://localhost:8080/health -> health_check() -> result

Instead of python calling our fucntion, someone over the internet can call it

#### What is an endpoint?

Each endpoint performs one task. Every endpoint is a URL connected to python function

#### Json is a universal format for exchanging data

almost every api in the world uses json

## Step 3 — Health Check Endpoint

Every production API has a health check — a simple endpoint that confirms the server is running. Load balancers and monitoring tools ping this constantly.

In [ ]:
#This creates a health-check endpoint to verify that your API is working.
@app.get("/health") #Creates a GET endpoint at /health.
def health_check():
    return {"status": "healthy", "version": "1.0.0"}

## Step 4 — Your First GET Endpoint

In [ ]:
@app.get("/")
def root():
    return {"message": "Welcome to the Enterprise GenAI API"}

@app.get("/students")
def get_students():
    return {
        "students": [
            {"id": 1, "name": "Rahul", "course": "Python"},
            {"id": 2, "name": "Priya", "course": "SQL"}
        ]
    }

In [ ]:
# Create a GET endpoint with a dynamic name
# {name} is a path parameter
# Example: /hello/Tulika
@app.get("/hello/{name}")
def hello(name: str):
      # The value from the URL is passed into the "name" parameter
    return {"message": f"Hello {name}"}

## Step 5 — Run the Server

localhost:8080/health (browser)->uvicorn -> fastapi -> run our python function ->  fastapi creates a response -> uvicorn -> browser

In [ ]:
What is host?

which ip address server shd bind to

127.0.0.1 -> localhost address

programs which are running on the same computer can access the server

0.0.0.0

listen to all available networks

my computer has

127.0.0.1 (localhost)
192......(wifi)


In [ ]:
import threading

#Normally, python executes one line at a time

# main notebook -> thread1 -> notebook cell -> thread 2 is executing....

#2 tasks -> 1. server to start and 2. notebook to keep running

def run_server():
    #start web server and serve this fastapi application
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

thread = threading.Thread(target=run_server, daemon=True) #creating a thread here
#target=run_server -> when this thread starts ...run run_server function
thread.start()

import time
time.sleep(2)
print("Server running at http://localhost:8000")
print("API docs at http://localhost:8000/docs")

ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8000): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted


Server running at http://localhost:8000
API docs at http://localhost:8000/docs


## Step 6 — Test the Endpoints

In [ ]:
import httpx  # Python library used to send HTTP requests

response = httpx.get("http://localhost:8000/health")# Send a GET request to the FastAPI /health endpoint
response.json() # Convert the JSON response from the API into a Python dictionary

{'status': 'healthy', 'version': '1.0.0'}

In [ ]:
response = httpx.get("http://localhost:8000/students")
response.json()

{'students': [{'id': 1, 'name': 'Rahul', 'course': 'Python'},
  {'id': 2, 'name': 'Priya', 'course': 'SQL'}]}

---
# Part 4 — Request & Response Models with Pydantic

## Why We Need This

Without validation, your API accepts anything — empty strings, wrong types, missing fields. This causes hard-to-debug errors deep inside your code.

Pydantic models define exactly what a request must look like. FastAPI validates every incoming request against the model automatically. If the request is wrong, FastAPI returns a clear error before your code even runs.

```python
class ChatRequest(BaseModel):
    message: str        ← required, must be a string
    model: str = "gpt-4o-mini"   ← optional, has a default
    max_tokens: int = 500        ← optional, has a default
```

If someone sends a request without `message`, FastAPI immediately returns a 422 error with a clear explanation — without you writing any validation code.


## Step 7 — Define Request and Response Models

In [ ]:
## Define the structure of the incoming request
class StudentRequest(BaseModel): #BaseModel is parent class provided by Pydantic
    name: str                 # Name of the student; name is required and must be a string.
    course: str               # Course selected by the student; required and must be a string.
    age: int                  # Age of the student; required and must be an integer.
    email: str                # Email of the student; required and must be a string.
    experience: Optional[int] = None  # Years of experience; Optional means this field can be None.

# Define the structure of the API response
class StudentResponse(BaseModel):
    student_id: int           # Unique ID assigned to the student
    message: str              # Confirmation message
    name: str                 # Name of the registered student
    course: str               # Course selected by the student

print("Models defined")


Models defined


## Step 8 — A POST Endpoint with Validation

In [ ]:
#unlike get, post requests include a request body
@app.post("/students", response_model=StudentResponse)
def register_student(request: StudentRequest):
    # FastAPI automatically validates the incoming request against StudentRequest
    # If name, course, age or email is missing or has the wrong type,
    # FastAPI returns 422 before reaching here

    # For now return a mock response — later we can connect a real database
    return StudentResponse(
        student_id=101,
        message="Student registered successfully",
        name=request.name,
        course=request.course
    )

## Step 9 — Test the POST Endpoint

In [ ]:
#Test valid request
import httpx

response = httpx.post(
    "http://localhost:8000/students",
    json={
        "name": "Rahul",
        "course": "Python",
        "age": 22,
        "email": "rahul@gmail.com"
    }
)

print("Status code:", response.status_code)
print(response.json())

Status code: 200
{'student_id': 101, 'message': 'Student registered successfully', 'name': 'Rahul', 'course': 'Python'}


In [ ]:
# Test validation — send a bad request (missing message)
# Test validation — send a bad request (missing name)
response = httpx.post(
    "http://localhost:8000/students",
    json={
        "course": "Python",
        "age": 22,
        "email": "rahul@gmail.com"
    }
)

print("Status code:", response.status_code)
print(response.json())

Status code: 422
{'detail': [{'type': 'missing', 'loc': ['body', 'name'], 'msg': 'Field required', 'input': {'course': 'Python', 'age': 22, 'email': 'rahul@gmail.com'}}]}


Notice the 422 status code and the clear error message — FastAPI handled this automatically with no extra code from us.

---
# Part 5 — Error Handling

## Why This Matters

Things go wrong in production — API keys expire, the LLM service goes down, requests time out. Without proper error handling, your users get confusing 500 errors with Python stack traces.

FastAPI's `HTTPException` lets you return clean, informative errors with the right HTTP status code.

| Status Code | Meaning |
|-------------|---------|
| 200 | Success |
| 400 | Bad request — client sent something wrong |
| 401 | Unauthorized — missing or invalid API key |
| 404 | Not found |
| 422 | Validation error — wrong data types |
| 500 | Server error — something broke on our side |


## Add Error Handling

In [ ]:
#student registration
@app.post("/students/safe", response_model=StudentResponse)
def register_student_safe(request: StudentRequest):

    if not request.name.strip():  # if name is empty
        raise HTTPException(status_code=400, detail="Name cannot be empty")

    if request.age < 18:
        raise HTTPException(status_code=400, detail="Student must be 18 or older")

    try:
        # For now, return a mock response
        return StudentResponse(
            student_id=101,
            message="Student registered successfully",
            name=request.name,
            course=request.course
        )

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Error registering student: {str(e)}"
        )

In [ ]:
# Test with empty name
response = httpx.post(
    "http://127.0.0.1:8000/students/safe",
    json={
        "name": "",
        "course": "Python",
        "age": 22,
        "email": "rahul@gmail.com"
    }
)

print("Status:", response.status_code)
print(response.json())

Status: 400
{'detail': 'Name cannot be empty'}


---
# Part 6 — Streaming Responses

## Why Streaming Matters?

Normally, an API prepares the complete response first and then sends it to the user.

For example, if an API needs to generate or retrieve a large amount of data, the user may have to wait until all the data is ready.

With streaming, the API sends the response piece by piece as the data becomes available.

For example:

**Without Streaming:**

API → Wait → Wait → Wait → Complete Response

**With Streaming:**

API → Data 1 → Data 2 → Data 3 → Data 4 → ...

This is useful when:

The response is large

Data takes time to generate

We want the user to start receiving data immediately

We don't want to wait for the entire response to be ready


## Streaming Endpoint

In [ ]:
#instead of waiting 5 seconds and returning all 5 students, the API sends one student every second.
from fastapi.responses import StreamingResponse
import time

@app.get("/students/stream")
def stream_students():

    def generate_students():
        for i in range(1, 6):
            yield f"Student {i}\n"
            time.sleep(1)

    return StreamingResponse(
        generate_students(),
        media_type="text/plain"
    )

In [ ]:
import httpx

with httpx.stream(
    "GET",
    "http://127.0.0.1:8000/students/stream"
) as response:

    for chunk in response.iter_text():
        print(chunk, end="")

Student 1
Student 2
Student 3
Student 4
Student 5
